In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

# Step 1: Get REAL carbon intensity data with pagination
def get_uk_carbon_intensity(start_date, end_date, chunk_days=7):
    """Fetch real UK grid carbon intensity from National Grid ESO with chunking"""
    all_data = []
    
    current_start = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    
    while current_start < end_date:
        current_end = min(current_start + timedelta(days=chunk_days), end_date)
        
        # Format dates for API
        from_str = current_start.strftime('%Y-%m-%dT%H:%MZ')
        to_str = current_end.strftime('%Y-%m-%dT%H:%MZ')
        
        url = f"https://api.carbonintensity.org.uk/intensity/{from_str}/{to_str}"
        print(f"Fetching data from {from_str} to {to_str}")
        
        response = requests.get(url)
        
        if response.status_code == 200:
            data = response.json()
            if 'data' in data and data['data']:
                all_data.extend(data['data'])
            time.sleep(0.5)  # Be nice to the API
        else:
            print(f"Warning: API error {response.status_code} for {from_str} to {to_str}")
            print(f"Response: {response.text}")
            # Continue with what we have
        
        current_start = current_end
    
    if not all_data:
        raise Exception("No data fetched from API")
    
    df = pd.DataFrame(all_data)
    df['from'] = pd.to_datetime(df['from'])
    df['intensity'] = df['intensity'].apply(lambda x: x['actual'] if 'actual' in x else x['forecast'])
    return df[['from', 'intensity']]

# Alternative: Use a smaller, realistic date range
def get_sample_carbon_data():
    """Get a smaller sample for testing"""
    # Get data for a specific month instead of huge range
    start_date = '2024-05-01'
    end_date = '2024-12-01'  # One month is usually safe
    
    print(f"Fetching carbon intensity data for {start_date} to {end_date}")
    return get_uk_carbon_intensity(start_date, end_date)

# Step 2: Map to your 48 half-hour periods
try:
    baseline = pd.read_csv(r"C:\Users\majid\OneDrive\FlexiBid_DA_Optimization_Engine_for_EV_Flexibility\data\baseline_profile.csv")
    
    # Ensure required columns exist
    if 'ptu_index' not in baseline.columns:
        # If ptu_index doesn't exist, create it from row index
        baseline['ptu_index'] = baseline.index
    
    baseline['hour'] = baseline['ptu_index'] // 2
    
    # Get carbon data for your analysis period (using smaller range)
    try:
        carbon_data = get_sample_carbon_data()
    except Exception as e:
        print(f"Could not fetch real data: {e}")
        print("Using sample carbon intensity data...")
        # Create synthetic carbon data for demonstration
        hours = list(range(24))
        # Typical UK carbon intensity pattern (lower at night, higher during day)
        sample_intensity = [180 + 20 * abs(h - 12) for h in hours]  # Peaks around noon
        hourly_carbon = pd.Series(sample_intensity, index=hours)
    else:
        # Calculate average carbon intensity by hour from real data
        carbon_data['hour'] = carbon_data['from'].dt.hour
        hourly_carbon = carbon_data.groupby('hour')['intensity'].mean()
    
    # Map to your baseline
    baseline['carbon_intensity'] = baseline['hour'].map(hourly_carbon)
    
    # Fill any missing values with average
    if baseline['carbon_intensity'].isna().any():
        avg_intensity = baseline['carbon_intensity'].mean()
        baseline['carbon_intensity'].fillna(avg_intensity, inplace=True)
    
    # Step 3: Calculate emissions
    # Assuming baseline_kw is in kW (power), convert to energy for half-hour period
    baseline['energy_kwh'] = baseline['baseline_kw'] * 0.5  # kW * 0.5h = kWh for half-hour
    baseline['emissions_kg'] = baseline['energy_kwh'] * baseline['carbon_intensity'] / 1000
    
    baseline_total = baseline['emissions_kg'].sum()
    
    print(f"\n=== Emissions Calculation ===")
    print(f"Data period covers {len(baseline)} half-hour periods ({len(baseline)/48:.1f} days)")
    print(f"Average carbon intensity: {baseline['carbon_intensity'].mean():.1f} gCO2/kWh")
    print(f"Total energy consumed: {baseline['energy_kwh'].sum():.2f} kWh")
    print(f"Baseline emissions: {baseline_total:.2f} kg CO2e")
    
    # For demonstration, create an "optimized" scenario (shift load to lower carbon times)
    if 'optimized_kw' not in baseline.columns:
        print("\nCreating optimized scenario (shifting 20% of load to lower-carbon hours)...")
        # Simple optimization: shift load from high to low carbon hours
        baseline['optimized_kw'] = baseline['baseline_kw'].copy()
        
        # Find high carbon hours (above average)
        avg_carbon = baseline['carbon_intensity'].mean()
        high_carbon_mask = baseline['carbon_intensity'] > avg_carbon
        
        # Reduce high-carbon load by 20%
        baseline.loc[high_carbon_mask, 'optimized_kw'] *= 0.8
        
        # Increase low-carbon load proportionally to maintain total energy
        total_reduction = baseline.loc[high_carbon_mask, 'baseline_kw'].sum() * 0.2
        low_carbon_mask = baseline['carbon_intensity'] <= avg_carbon
        if low_carbon_mask.any():
            increase_factor = 1 + (total_reduction / baseline.loc[low_carbon_mask, 'baseline_kw'].sum())
            baseline.loc[low_carbon_mask, 'optimized_kw'] *= increase_factor
    
    # Calculate optimized emissions
    baseline['optimized_energy_kwh'] = baseline['optimized_kw'] * 0.5
    baseline['optimized_emissions_kg'] = baseline['optimized_energy_kwh'] * baseline['carbon_intensity'] / 1000
    optimized_total = baseline['optimized_emissions_kg'].sum()
    
    # Step 4: Report
    print(f"\n=== Results ===")
    print(f"Baseline emissions: {baseline_total:.2f} kg CO2e")
    print(f"Optimized emissions: {optimized_total:.2f} kg CO2e")
    print(f"Reduction: {baseline_total - optimized_total:.2f} kg CO2e ({((baseline_total - optimized_total)/baseline_total*100):.1f}%)")
    print(f"\n=== Energy Summary ===")
    print(f"Baseline energy: {baseline['energy_kwh'].sum():.2f} kWh")
    print(f"Optimized energy: {baseline['optimized_energy_kwh'].sum():.2f} kWh")
    
    # Save results
    baseline.to_csv('emissions_calculation_results.csv', index=False)
    print(f"\nResults saved to 'emissions_calculation_results.csv'")
    
except FileNotFoundError:
    print("Error: Could not find baseline_profile.csv at the specified path")
    print("Creating sample data for demonstration...")
    
    # Create sample data
    baseline = pd.DataFrame({
        'ptu_index': range(48),
        'baseline_kw': [50 + 20 * abs((i//2) - 12) for i in range(48)]  # Simulated load profile
    })
    baseline['hour'] = baseline['ptu_index'] // 2
    
    # Create sample carbon data
    hours = list(range(24))
    sample_intensity = [180 + 20 * abs(h - 12) for h in hours]
    hourly_carbon = pd.Series(sample_intensity, index=hours)
    baseline['carbon_intensity'] = baseline['hour'].map(hourly_carbon)
    
    # Calculate emissions
    baseline['energy_kwh'] = baseline['baseline_kw'] * 0.5
    baseline['emissions_kg'] = baseline['energy_kwh'] * baseline['carbon_intensity'] / 1000
    baseline_total = baseline['emissions_kg'].sum()
    
    print(f"\nUsing sample data (48 half-hour periods)")
    print(f"Baseline emissions: {baseline_total:.2f} kg CO2e")

Fetching carbon intensity data for 2024-05-01 to 2024-12-01
Fetching data from 2024-05-01T00:00Z to 2024-05-08T00:00Z
Fetching data from 2024-05-08T00:00Z to 2024-05-15T00:00Z
Fetching data from 2024-05-15T00:00Z to 2024-05-22T00:00Z
Fetching data from 2024-05-22T00:00Z to 2024-05-29T00:00Z
Fetching data from 2024-05-29T00:00Z to 2024-06-05T00:00Z
Fetching data from 2024-06-05T00:00Z to 2024-06-12T00:00Z
Fetching data from 2024-06-12T00:00Z to 2024-06-19T00:00Z
Fetching data from 2024-06-19T00:00Z to 2024-06-26T00:00Z
Fetching data from 2024-06-26T00:00Z to 2024-07-03T00:00Z
Fetching data from 2024-07-03T00:00Z to 2024-07-10T00:00Z
Fetching data from 2024-07-10T00:00Z to 2024-07-17T00:00Z
Fetching data from 2024-07-17T00:00Z to 2024-07-24T00:00Z
Fetching data from 2024-07-24T00:00Z to 2024-07-31T00:00Z
Fetching data from 2024-07-31T00:00Z to 2024-08-07T00:00Z
Fetching data from 2024-08-07T00:00Z to 2024-08-14T00:00Z
Fetching data from 2024-08-14T00:00Z to 2024-08-21T00:00Z
Fetching dat